# DataFrame Validation with Pandera

In the [Combining through Concat and Merge](06_combine.ipynb) lesson, we merged US population data with state abbreviations and areas using `how='left'`. The merge **succeeded** — but some rows ended up with missing values that nobody checked.

**Learning objectives:**

- Explain why validating dataframe structure and values matters after modify/combine steps
- Install and use [pandera](https://pandera.readthedocs.io/en/stable/index.html) with pandas DataFrames
- Read basic Python **type hints** and use them in a `DataFrameModel`
- Validate real data from the modify and combine lessons, catch errors, and read error reports
- Write custom checks for domain-specific rules (medical/statistical)


## Why validate?

After reading, modifying, and combining data, silent problems are common:

- **Wrong dtypes** — numbers stored as strings after `read_csv`
- **Out-of-range values** — a negative height or impossible BMI
- **Silent `NaN`s from merges** — a left join keeps every row, even when there is no match
- **Unexpected categories** — a typo like `"Normol"` instead of `"Normal"`

Pandas will happily compute on bad data. The result is **garbage in, garbage out**. Validation gives you an explicit **contract** for what each column should look like *before* you analyze or visualize.


## What is pandera?

> *Data validation for scientists, engineers, and analysts seeking correctness.*

[Pandera](https://pandera.readthedocs.io/en/stable/index.html) is an open-source library that checks whether a dataframe matches a **schema** — a description of expected columns, types, and value rules.

**What you can do with pandera:**

1. Define a schema once and validate pandas (and other) dataframes
2. Check column **types** and **properties** (e.g. values between 0 and 100)
3. Run **lazy validation** to collect *all* errors in one report
4. Add **custom checks** for business or scientific rules
5. Attach validation to functions with **decorators** (optional, advanced)

**Note on pydantic:** You may see pandera described as "pydantic-style." [Pydantic](https://docs.pydantic.dev/latest/) is a popular library for validating ordinary Python objects (e.g. a user's email or age). Pandera brings that same *idea* to tabular data. You do **not** need to know pydantic to use this lesson.


## Supported backends

Pandera supports several dataframe libraries. This course uses **pandas**.

| Backend | Used in this lesson? | Notes |
|---------|----------------------|-------|
| **pandas** | Yes | Most features; what we use below |
| polars | No | Fast alternative dataframe library |
| pyspark | No | Big-data / distributed workloads |
| ibis | No | SQL-like analytics across engines |

See the full [feature matrix by backend](https://pandera.readthedocs.io/en/stable/index.html#supported-features-by-dataframe-backend) in the docs.


## Installation

Install pandera with the pandas extra:

```bash
uv add 'pandera[pandas]'
```

## A quick detour: Python type hints

Pandera's **DataFrame Models** use **type hints** — optional labels on variables and function parameters that describe what type of value is expected.

Python **does not enforce** these labels at runtime by itself. They are documentation that tools (and pandera) can use.

**Example — function hints:**

```python
def to_kg(pounds: float) -> float:
    """Convert weight from pounds to kilograms."""
    return pounds * 0.453592
```

- `pounds: float` — the input should be a floating-point number
- `-> float` — the function returns a float

**Example — variable hint:**

```python
age: int = 30  # age is intended to be an integer
```

Pandera reads these hints on a `DataFrameModel` class and **enforces** them when you call `.validate()`.


## DataFrame Models: a basic example

The [DataFrame Models](https://pandera.readthedocs.io/en/stable/dataframe_models.html) API lets you define a schema as a Python class. Each **attribute** is a column; its **type hint** is the expected dtype; `pa.Field(...)` adds extra rules.

We'll start with a small **monthly sales** dataset (business context):


In [1]:
import pandas as pd
import pandera.pandas as pa  # https://pandera.readthedocs.io/en/stable/
from pandera.typing.pandas import Series


In [2]:
class MonthlySalesSchema(pa.DataFrameModel):
    """Schema for monthly store revenue records."""

    year: Series[int] = pa.Field(
        gt=2000,       # year must be after 2000
        coerce=True,   # try to convert strings like "2024" to int
    )
    month: Series[int] = pa.Field(
        ge=1,          # month >= 1
        le=12,         # month <= 12
        coerce=True,
    )
    revenue: Series[float] = pa.Field(
        ge=0,          # revenue cannot be negative
    )


In [3]:
good_sales = pd.DataFrame({
    "year": [2023, 2024, 2024],
    "month": [11, 1, 6],
    "revenue": [12500.0, 9800.5, 14300.0],
})

MonthlySalesSchema.validate(good_sales, lazy=True)


,year,month,revenue
0,2023,11,12500.0
1,2024,1,9800.5
2,2024,6,14300.0


Validation passed. Pandera coerced integer columns and checked value ranges.

Now introduce a bad row — year before 2000:


In [4]:
bad_sales = pd.DataFrame({
    "year": [2023, 1999],
    "month": [11, 6],
    "revenue": [12500.0, 500.0],
})

try:
    MonthlySalesSchema.validate(bad_sales, lazy=True)
except pa.errors.SchemaErrors as exc:
    print(exc)


{
    "DATA": {
        "DATAFRAME_CHECK": [
            {
                "schema": "MonthlySalesSchema",
                "column": "year",
                "check": "greater_than(2000)",
                "error": "Column 'year' failed element-wise validator number 0: greater_than(2000) failure cases: 1999"
            }
        ]
    }
}


The error names the **column** (`year`), the **check** (`greater_than(2000)`), and the **failure cases** (`1999`).

---

## Validating the BMI dataframe (from the modify lesson)

We rebuild the athlete BMI table from [Creating and modifying columns](05_modify.ipynb), then define a schema for it.


In [5]:
from pathlib import Path

data_dir = Path("../data")
data_socr = data_dir / "SOCR-HeightWeight.csv"

df_socr = pd.read_csv(data_socr, index_col="Index")
df_socr["Height(m)"] = df_socr["Height(Inches)"] * 0.0254
df_socr["Weight(kg)"] = df_socr["Weight(Pounds)"] * 0.453592
df_socr["BMI"] = df_socr["Weight(kg)"] / (df_socr["Height(m)"] ** 2)


def get_bmi_category(bmi: float) -> str:
    if bmi < 16:
        return "Severe Thinness"
    elif bmi < 17:
        return "Moderate Thinness"
    elif bmi < 18.5:
        return "Mild Thinness"
    elif bmi < 25:
        return "Normal"
    elif bmi < 30:
        return "Overweight"
    elif bmi < 35:
        return "Obese Class I"
    elif bmi < 40:
        return "Obese Class II"
    else:
        return "Obese Class III"


df_socr["classification"] = df_socr["BMI"].apply(get_bmi_category)
df_socr = df_socr.drop(columns=["Height(Inches)", "Weight(Pounds)"])
df_socr.head()


,Height(m),Weight(kg),BMI,classification
Index,,,,
1,1.670896,51.252494,18.357609,Mild Thinness
2,1.816486,61.909547,18.762615,Normal
3,1.762728,69.411778,22.338940,Normal
4,1.732702,64.562199,21.504569,Normal
5,1.721810,65.452010,22.077625,Normal


In [6]:
WHO_CATEGORIES = [
    "Severe Thinness",
    "Moderate Thinness",
    "Mild Thinness",
    "Normal",
    "Overweight",
    "Obese Class I",
    "Obese Class II",
    "Obese Class III",
]


class BmiSchema(pa.DataFrameModel):
    """Schema for height/weight/BMI records (WHO categories)."""

    Height_m: Series[float] = pa.Field(
        alias="Height(m)",
        gt=0,
    )
    Weight_kg: Series[float] = pa.Field(
        alias="Weight(kg)",
        gt=0,
    )
    BMI: Series[float] = pa.Field(
        gt=0,
    )
    classification: Series[str] = pa.Field(
        isin=WHO_CATEGORIES,
    )


In [7]:
# Reset index so column names match the schema (index is not validated here)
df_bmi = df_socr.reset_index(drop=True)
BmiSchema.validate(df_bmi, lazy=True)


,Height(m),Weight(kg),BMI,classification
0,1.670896,51.252494,18.357609,Mild Thinness
1,1.816486,61.909547,18.762615,Normal
2,1.762728,69.411778,22.338940,Normal
3,1.732702,64.562199,21.504569,Normal
4,1.721810,65.452010,22.077625,Normal
...,...,...,...,...
24995,1.765355,53.538008,17.179016,Mild Thinness
24996,1.639526,54.518674,20.281906,Normal
24997,1.643343,53.644285,19.864010,Normal
24998,1.715241,59.995797,20.392499,Normal


Clean data passes. Next we **deliberately corrupt** a copy to see how pandera reports problems.

## Introducing a deliberate error

Real-world data entry mistakes might include a negative height or a typo in a category label.


In [8]:
df_bmi_bad = df_bmi.copy()
df_bmi_bad.loc[0, "Height(m)"] = -1.5          # impossible height
df_bmi_bad.loc[1, "classification"] = "Normol"  # typo for "Normal"

try:
    BmiSchema.validate(df_bmi_bad, lazy=True)
except pa.errors.SchemaErrors as exc:
    print("--- Error message (human-readable summary) ---")
    print(exc)


--- Error message (human-readable summary) ---
{
    "DATA": {
        "DATAFRAME_CHECK": [
            {
                "schema": "BmiSchema",
                "column": "Height(m)",
                "check": "greater_than(0)",
                "error": "Column 'Height(m)' failed element-wise validator number 0: greater_than(0) failure cases: -1.5"
            },
            {
                "schema": "BmiSchema",
                "column": "classification",
                "check": "isin(['Severe Thinness', 'Moderate Thinness', 'Mild Thinness', 'Normal', 'Overweight', 'Obese Class I', 'Obese Class II', 'Obese Class III'])",
                "error": "Column 'classification' failed element-wise validator number 0: isin(['Severe Thinness', 'Moderate Thinness', 'Mild Thinness', 'Normal', 'Overweight', 'Obese Class I', 'Obese Class II', 'Obese Class III']) failure cases: Normol"
            }
        ]
    }
}


**How to read the message:**

- **Column** — which field failed (`Height(m)` or `classification`)
- **Check** — the rule (`greater_than(0)` or `isin([...])`)
- **failure cases** — the actual bad values (`-1.5`, `Normol`)

## Lazy validation

By default, validation can stop at the **first** error. With **`lazy=True`**, pandera runs **all** checks and raises a single `SchemaErrors` exception listing every problem — much easier when debugging multiple columns.

See [Lazy Validation](https://pandera.readthedocs.io/en/stable/lazy_validation.html).

We recommend **`lazy=True` by default** so you see the full picture.


In [9]:
df_bmi_multi_bad = df_bmi.copy()
df_bmi_multi_bad.loc[0, "Height(m)"] = -1.5
df_bmi_multi_bad.loc[1, "Weight(kg)"] = -10.0
df_bmi_multi_bad.loc[2, "classification"] = "Normol"
df_bmi_multi_bad.loc[3, "BMI"] = 999.0  # implausible BMI

try:
    BmiSchema.validate(df_bmi_multi_bad, lazy=True)
except pa.errors.SchemaErrors as exc:
    print(f"Number of failure rows in report: {len(exc.failure_cases)}")
    display(exc.failure_cases)


Number of failure rows in report: 3


,schema_context,column,check,check_number,failure_case,index
0,Column,Height(m),greater_than(0),0,-1.5,0
1,Column,Weight(kg),greater_than(0),0,-10.0,1
2,Column,classification,"isin(['Severe Thinness', 'Moderate Thinness', ...",0,Normol,2


## Error reports

With `lazy=True`, pandera aggregates errors into a structured [error report](https://pandera.readthedocs.io/en/stable/error_report.html). Errors are grouped under **SCHEMA** (structure/types) and **DATA** (value checks).


In [10]:
import json

try:
    BmiSchema.validate(df_bmi_multi_bad, lazy=True)
except pa.errors.SchemaErrors as exc:
    print(json.dumps(exc.message, indent=2))


{
  "DATA": {
    "DATAFRAME_CHECK": [
      {
        "schema": "BmiSchema",
        "column": "Height(m)",
        "check": "greater_than(0)",
        "error": "Column 'Height(m)' failed element-wise validator number 0: greater_than(0) failure cases: -1.5"
      },
      {
        "schema": "BmiSchema",
        "column": "Weight(kg)",
        "check": "greater_than(0)",
        "error": "Column 'Weight(kg)' failed element-wise validator number 0: greater_than(0) failure cases: -10.0"
      },
      {
        "schema": "BmiSchema",
        "column": "classification",
        "check": "isin(['Severe Thinness', 'Moderate Thinness', 'Mild Thinness', 'Normal', 'Overweight', 'Obese Class I', 'Obese Class II', 'Obese Class III'])",
        "error": "Column 'classification' failed element-wise validator number 0: isin(['Severe Thinness', 'Moderate Thinness', 'Mild Thinness', 'Normal', 'Overweight', 'Obese Class I', 'Obese Class II', 'Obese Class III']) failure cases: Normol"
      }
    ]
  

The JSON report is machine-readable — useful in pipelines. For interactive debugging, `exc.failure_cases` is a tidy table of every failing value.

---

## Validating the US states dataframe (from the combine lesson)

We rebuild `final_df` from [Combining through Concat and Merge](06_combine.ipynb) and validate it. This is where validation pays off: the left joins **looked fine**, but some regions never matched.


In [11]:
pop_url = (
    "https://raw.githubusercontent.com/jakevdp/data-USstates/master/"
    "state-population.csv"
)
abbrev_url = (
    "https://raw.githubusercontent.com/jakevdp/data-USstates/master/"
    "state-abbrevs.csv"
)
areas_url = (
    "https://raw.githubusercontent.com/jakevdp/data-USstates/master/"
    "state-areas.csv"
)

df_pop_raw = pd.read_csv(pop_url)
df_abbrevs = pd.read_csv(abbrev_url)
df_areas = pd.read_csv(areas_url)

pop_1990s = df_pop_raw[df_pop_raw["year"] < 2000]
pop_2000s = df_pop_raw[df_pop_raw["year"] >= 2000]
pop_full = pd.concat([pop_1990s, pop_2000s])

merged = pd.merge(
    pop_full,
    df_abbrevs,
    how="left",
    left_on="state/region",
    right_on="abbreviation",
)
merged = merged.drop(columns=["abbreviation"])

final_df = pd.merge(
    merged,
    df_areas,
    how="left",
    on="state",
)
final_df.head()


,state/region,ages,year,population,state,area (sq. mi)
0,AL,under18,1999,1121287.0,Alabama,52423.0
1,AL,total,1999,4430141.0,Alabama,52423.0
2,AL,total,1998,4404701.0,Alabama,52423.0
3,AL,under18,1998,1118252.0,Alabama,52423.0
4,AL,under18,1997,1122893.0,Alabama,52423.0


In [12]:
class StatePopulationSchema(pa.DataFrameModel):
    """Schema for merged US state population + area records."""

    state_region: Series[str] = pa.Field(
        alias="state/region",
        str_length={"min_value": 2, "max_value": 3},  # state abbreviation codes
    )
    ages: Series[str] = pa.Field(
        isin=["under18", "total"],
    )
    year: Series[int] = pa.Field(
        ge=1990,
        le=2013,
    )
    population: Series[float] = pa.Field(
        gt=0,
        nullable=False,
    )
    state: Series[str] = pa.Field(
        nullable=False,  # every row should have matched a full state name
    )
    area_sq_mi: Series[float] = pa.Field(
        alias="area (sq. mi)",
        gt=0,
        nullable=False,
    )


In [13]:
try:
    StatePopulationSchema.validate(final_df, lazy=True)
except pa.errors.SchemaErrors as exc:
    print("Validation failed — as expected for rows that did not match a US state.")
    print()
    print("Regions with missing state names:")
    print(
        final_df.loc[final_df["state"].isna(), "state/region"]
        .drop_duplicates()
        .tolist()
    )
    print()
    print(json.dumps(exc.message, indent=2)[:1500], "...")


Validation failed — as expected for rows that did not match a US state.

Regions with missing state names:
['PR', 'USA']

{
  "SCHEMA": {
    "SERIES_CONTAINS_NULLS": [
      {
        "schema": "StatePopulationSchema",
        "column": "population",
        "check": "not_nullable",
        "error": "non-nullable series 'population' contains null values:1020   NaN1021   NaN1022   NaN1023   NaN1024   NaN1025   NaN1026   NaN1027   NaN1028   NaN1029   NaN1030   NaN1031   NaN1032   NaN1033   NaN1034   NaN1035   NaN1036   NaN1037   NaN1038   NaN1039   NaNName: population, dtype: float64"
      },
      {
        "schema": "StatePopulationSchema",
        "column": "state",
        "check": "not_nullable",
        "error": "non-nullable series 'state' contains null values:1020    NaN1021    NaN1022    NaN1023    NaN1024    NaN       ... 2539    NaN2540    NaN2541    NaN2542    NaN2543    NaNName: state, Length: 96, dtype: str"
      },
      {
        "schema": "StatePopulationSchema",
    

**What happened?** Codes like `USA` (national total) and `PR` (Puerto Rico) appear in the population file but are **not** in the state abbreviations lookup. The left join kept those rows but filled `state` and `area (sq. mi)` with `NaN`.

**What to do:** Filter them out, add them to the lookup table, or relax the schema (e.g. `nullable=True`) if those rows are intentional.


In [14]:
# Keep only rows that matched a US state (drop USA / PR aggregates for this analysis)
states_only = final_df.dropna(subset=["state", "area (sq. mi)"])
StatePopulationSchema.validate(states_only, lazy=True)
print(f"Validated {len(states_only):,} rows successfully.")


Validated 2,448 rows successfully.


## Custom checks

Built-in `Field` rules cover many cases. For domain logic — e.g. plausible human BMI or consistency between columns — use [custom checks](https://pandera.readthedocs.io/en/stable/dataframe_models.html#custom-checks) as class methods.


In [15]:
class BmiSchemaStrict(BmiSchema):
    """BMI schema with medical plausibility and formula consistency checks."""

    @pa.check("BMI", name="plausible_bmi")
    def bmi_in_human_range(cls, bmi: pd.Series) -> pd.Series:
        # WHO-relevant range for adults; catches typos and unit errors
        return (bmi >= 10) & (bmi <= 60)

    @pa.dataframe_check(name="bmi_formula_consistent")
    def bmi_matches_formula(cls, df: pd.DataFrame) -> pd.Series:
        expected = df["Weight(kg)"] / (df["Height(m)"] ** 2)
        return (df["BMI"] - expected).abs() < 0.01


In [16]:
BmiSchemaStrict.validate(df_bmi, lazy=True)
print("Clean BMI data passes custom checks.")


Clean BMI data passes custom checks.


In [17]:
df_bmi_tampered = df_bmi.copy()
df_bmi_tampered.loc[0, "BMI"] = 50.0  # does not match height/weight formula

try:
    BmiSchemaStrict.validate(df_bmi_tampered, lazy=True)
except pa.errors.SchemaErrors as exc:
    print(exc)


{
    "DATA": {
        "DATAFRAME_CHECK": [
            {
                "schema": "BmiSchemaStrict",
                "column": "BmiSchemaStrict",
                "check": "bmi_formula_consistent",
                "error": "DataFrameSchema 'BmiSchemaStrict' failed element-wise validator number 0: <Check bmi_formula_consistent> failure cases: 1.6708960739999998, 51.252494060000004, 50.0, Mild Thinness"
            }
        ]
    }
}


## Key Takeaways

- **Schemas are contracts** — define expected columns, types, and rules once; validate after every transform.
- **Type hints** on a `DataFrameModel` tell pandera what each column should contain; `pa.Field(...)` adds numeric ranges, allowed categories, and more.
- Prefer **`lazy=True`** so one validation run reports **all** failures across columns.
- **Error reports** group SCHEMA vs DATA issues; use `exc.failure_cases` for row-level debugging.
- **Custom checks** encode domain rules (WHO BMI categories, merge completeness, formula consistency).
- Validation belongs **right after modify/combine** — before visualization and analysis.
